# Tests de mini-funcionalidades de OP-14 `import_traces` y OP-15 `validate_traces`

Este notebook se usa para probar helpers y bloques internos de `import_traces_from_dataframe()` y `validate_traces()` antes de pasar a pruebas más integradas de las funciones públicas.

Objetivo:
- verificar primero las utilidades internas que sostienen el contrato,
- luego probar los helpers principales de import traces,
- y finalmente probar los helpers principales de validate traces.


## Bloque 0. Preparación

### Imports generales

In [1]:
import json
from copy import deepcopy

import numpy as np
import pandas as pd

### Imports del módulo

In [2]:
from pylondrina.schema import FieldSpec, TraceSchema
from pylondrina.datasets import TraceDataset
from pylondrina.reports import Issue, ImportReport, ConsistencyReport
from pylondrina.errors import ImportError as PylondrinaImportError
from pylondrina.errors import ValidationError, SchemaError

from pylondrina.importing_traces import (
    ImportTraceOptions,
    _normalize_import_trace_options,
    _preflight_import_traces_request,
    _resolve_trace_import_columns,
    _materialize_trace_core,
    _normalize_trace_time_utc,
    _finalize_import_traces_result,
    _duplicated_columns,
    _normalize_timezone_spec,
    _build_issues_summary as _build_import_issues_summary,
    _json_safe_scalar as _json_safe_scalar_import,
    _json_safe as _json_safe_import,
    _json_is_serializable as _json_is_serializable_import,
)

# Si este import falla por el catálogo actual de issues, corrígelo primero y vuelve a ejecutar.
from pylondrina.validation_traces import (
    TraceValidationOptions,
    _normalize_trace_validation_options,
    _preflight_validate_traces_request,
    _resolve_trace_validation_targets,
    _check_trace_required_and_types,
    _check_trace_constraints,
    _check_trace_monotonic_time_per_user,
    _finalize_trace_validation,
    _is_valid_constraint_payload,
    _invalid_mask_for_dtype,
    _sample_rows,
    _sample_index_list,
    _sample_list as _sample_list_validate,
    _build_issues_summary as _build_validate_issues_summary,
    _json_safe_row,
)

### Helpers de apoyo para test

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")

def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj, default=str)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e

def codes(issues):
    return [issue.code for issue in issues]

def assert_has_code(issues, expected_code: str):
    found = any(issue.code == expected_code for issue in issues)
    if not found:
        raise AssertionError(f"No se encontró el code esperado: {expected_code}. Códigos observados: {codes(issues)}")

def make_trace_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
    )

def make_trace_schema(
    fields: list[FieldSpec],
    *,
    required: list[str] | None = None,
    timezone: str | None = None,
    crs: str | None = "EPSG:4326",
    version: str = "trace-1.1",
) -> TraceSchema:
    return TraceSchema(
        version=version,
        fields={f.name: f for f in fields},
        required=list(required or []),
        timezone=timezone,
        crs=crs,
    )

def make_trace_dataset(
    df: pd.DataFrame,
    schema: TraceSchema,
    *,
    metadata: dict | None = None,
    provenance: dict | None = None,
) -> TraceDataset:
    return TraceDataset(
        data=df.copy(deep=True),
        schema=schema,
        metadata=deepcopy(metadata or {}),
        provenance=deepcopy(provenance or {}),
    )

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

show_ok("Bloque 0 cargado")

OK - Bloque 0 cargado


### Test 1.1 - Helpers internos de `import_traces`

Qué prueba:
- `_duplicated_columns`
- `_normalize_timezone_spec`
- `_json_safe_scalar`, `_json_safe`, `_json_is_serializable`
- `_build_issues_summary`

La idea es asegurar que las utilidades base del import estén bien antes de probar el pipeline principal.

In [4]:
# _duplicated_columns
assert _duplicated_columns(["a", "b", "a", "c", "b", "a"]) == ["a", "b"]

# _normalize_timezone_spec
tz_obj, tz_kind = _normalize_timezone_spec("UTC")
assert tz_obj == "UTC"
assert tz_kind == "utc"

tz_obj, tz_kind = _normalize_timezone_spec("-03:00")
assert tz_kind == "offset"
assert tz_obj.utcoffset(None).total_seconds() == -3 * 3600

tz_obj, tz_kind = _normalize_timezone_spec("America/Santiago")
assert tz_kind == "iana"

tz_obj, tz_kind = _normalize_timezone_spec("not/a_real_timezone")
assert tz_kind == "invalid"

# _json_safe_scalar / _json_safe / _json_is_serializable
safe_int = _json_safe_scalar_import(np.int64(7))
safe_float = _json_safe_scalar_import(np.float64(2.5))
safe_bool = _json_safe_scalar_import(np.bool_(True))
safe_ts = _json_safe_scalar_import(pd.Timestamp("2026-01-01T08:00:00Z"))

assert safe_int == 7 and isinstance(safe_int, int)
assert safe_float == 2.5 and isinstance(safe_float, float)
assert safe_bool is True
assert isinstance(safe_ts, str)

payload = {
    "num": np.int64(5),
    "flag": np.bool_(False),
    "ts": pd.Timestamp("2026-01-01T08:00:00Z"),
    "vals": {np.int64(1), np.float64(2.5)},
}
payload_safe = _json_safe_import(payload)
assert_json_safe(payload_safe, "payload_safe_import")
assert _json_is_serializable_import(payload_safe) is True

# _build_issues_summary
issues_summary_input = [
    Issue(level="warning", code="IMP.TEST.WARN_A", message="warn A1"),
    Issue(level="warning", code="IMP.TEST.WARN_A", message="warn A2"),
    Issue(level="error", code="IMP.TEST.ERR_B", message="err B1"),
]
summary = _build_import_issues_summary(issues_summary_input)
assert summary["counts"] == {"info": 0, "warning": 2, "error": 1}
assert summary["top_codes"][0] == {"code": "IMP.TEST.WARN_A", "count": 2}
assert summary["top_codes"][1] == {"code": "IMP.TEST.ERR_B", "count": 1}

show_ok("Test 1.1 - helpers internos de import_traces")

OK - Test 1.1 - helpers internos de import_traces


### Test 1.2 - Helpers internos de `validate_traces`

Qué prueba:
- `_is_valid_constraint_payload`
- `_invalid_mask_for_dtype`
- `_sample_rows`
- `_sample_index_list`
- `_sample_list`
- `_json_safe_row`
- `_build_issues_summary`

Esto permite aislar problemas de validación básica antes de probar helpers más grandes de OP-15.

In [5]:
# _is_valid_constraint_payload
assert _is_valid_constraint_payload("string", "nullable", True) is True
assert _is_valid_constraint_payload("float", "range", {"min": 0, "max": 10}) is True
assert _is_valid_constraint_payload("datetime", "datetime", {"allow_naive": False, "timezone": "UTC"}) is True
assert _is_valid_constraint_payload("string", "pattern", r"^[A-Z]+$") is True
assert _is_valid_constraint_payload("string", "length", {"min": 1, "max": 5}) is True
assert _is_valid_constraint_payload("bool", "unique", {"value": True}) is True

assert _is_valid_constraint_payload("float", "range", {"minimum": 0}) is False
assert _is_valid_constraint_payload("datetime", "datetime", {"foo": "bar"}) is False
assert _is_valid_constraint_payload("string", "pattern", "[") is False

# _invalid_mask_for_dtype
mask_int, sample_int = _invalid_mask_for_dtype(pd.Series(["1", "2", "2.5", None]), "int")
assert mask_int.tolist() == [False, False, True, False]
assert sample_int == ["2.5"]

mask_float, sample_float = _invalid_mask_for_dtype(pd.Series(["1.2", "abc", None]), "float")
assert mask_float.tolist() == [False, True, False]
assert sample_float == ["abc"]

mask_dt, sample_dt = _invalid_mask_for_dtype(pd.Series(["2026-01-01", "not-a-date", None]), "datetime")
assert mask_dt.tolist() == [False, True, False]
assert sample_dt == ["not-a-date"]

mask_bool, sample_bool = _invalid_mask_for_dtype(pd.Series(["true", "0", "maybe", None]), "bool")
assert mask_bool.tolist() == [False, False, True, False]
assert sample_bool == ["maybe"]

# _sample_rows / _sample_index_list / _sample_list / _json_safe_row
df_sample = pd.DataFrame(
    {
        "a": [1, 2, 3],
        "b": ["x", "y", "z"],
        "ts": pd.to_datetime(["2026-01-01", "2026-01-02", "2026-01-03"]),
    },
    index=[10, 11, 12],
)
mask = pd.Series([True, False, True], index=df_sample.index)

rows = _sample_rows(df_sample, mask, limit=2)
assert len(rows) == 2
assert_json_safe(rows, "sample_rows")

idxs = _sample_index_list(df_sample.index, limit=2)
assert idxs == [10, 11]

vals = _sample_list_validate([np.int64(1), pd.Timestamp("2026-01-01"), None], limit=3)
assert_json_safe(vals, "sample_list_validate")

safe_row = _json_safe_row({"x": np.int64(1), "ts": pd.Timestamp("2026-01-01")})
assert_json_safe(safe_row, "json_safe_row")

# _build_issues_summary de validate
issues_validate = [
    Issue(level="warning", code="VAL.TEST.WARN_A", message="warn A1"),
    Issue(level="warning", code="VAL.TEST.WARN_A", message="warn A2"),
    Issue(level="error", code="VAL.TEST.ERR_B", message="err B1"),
]
summary = _build_validate_issues_summary(issues_validate)
assert summary["counts"] == {"info": 0, "warning": 2, "error": 1}
assert summary["top_codes"][0] == {"code": "VAL.TEST.WARN_A", "count": 2}
assert summary["top_codes"][1] == {"code": "VAL.TEST.ERR_B", "count": 1}

show_ok("Test 1.2 - helpers internos de validate_traces")

OK - Test 1.2 - helpers internos de validate_traces


## Bloque 2. Tests de helpers principales de `import_traces`

Qué se prueba en este bloque:
- normalización de options,
- preflight fatal,
- resolución de columnas,
- materialización del core mínimo,
- normalización de `time_utc`,
- y finalización del resultado.

Este bloque ya entra directamente a la lógica central de OP-14.

### Fixtures reutilizables mínimas para OP-14

In [6]:
BASE_IMPORT_FIELDS = [
    make_trace_field("point_id", "string", required=True, constraints={"unique": True}),
    make_trace_field("user_id", "string", required=True),
    make_trace_field("time_utc", "datetime", required=True),
    make_trace_field("latitude", "float", required=True, constraints={"range": {"min": -90, "max": 90}}),
    make_trace_field("longitude", "float", required=True, constraints={"range": {"min": -180, "max": 180}}),
    make_trace_field("location_category", "string", required=False, constraints={"length": {"max": 40}}),
]

BASE_IMPORT_SCHEMA = make_trace_schema(
    BASE_IMPORT_FIELDS,
    required=["point_id", "user_id", "time_utc", "latitude", "longitude"],
    timezone=None,
)

RAW_POINTS = pd.DataFrame(
    {
        "uid": ["u1", "u2"],
        "when": ["2026-01-01 08:00:00", "2026-01-01 09:30:00"],
        "lat": [-33.45, -33.46],
        "lon": [-70.66, -70.67],
        "poi_cat": ["home", "work"],
        "noise": ["x", "y"],
    }
)

RAW_TO_CANONICAL = {
    "user_id": "uid",
    "time_utc": "when",
    "latitude": "lat",
    "longitude": "lon",
    "location_category": "poi_cat",
}

show_ok("Fixtures de OP-14 listas")

OK - Fixtures de OP-14 listas


### Test 2.1 - `_normalize_import_trace_options`

Qué prueba:
- defaults cuando `options=None`,
- normalización de `selected_fields` a lista,
- y construcción de `parameters_effective` JSON-safe.

In [7]:
options_eff, params = _normalize_import_trace_options(None)
assert isinstance(options_eff, ImportTraceOptions)
assert options_eff.keep_extra_fields is True
assert options_eff.selected_fields is None
assert options_eff.strict is False
assert options_eff.source_timezone is None
assert params["selected_fields"] is None
assert_json_safe(params, "import_trace_parameters_default")

options_eff, params = _normalize_import_trace_options(
    ImportTraceOptions(
        keep_extra_fields=False,
        selected_fields=("location_category", "noise"),
        strict=True,
        source_timezone="-03:00",
    )
)
assert options_eff.keep_extra_fields is False
assert options_eff.selected_fields == ["location_category", "noise"]
assert options_eff.strict is True
assert options_eff.source_timezone == "-03:00"
assert params["selected_fields"] == ["location_category", "noise"]
assert params["strict"] is True
assert_json_safe(params, "import_trace_parameters_custom")

show_ok("Test 2.1 - _normalize_import_trace_options")

OK - Test 2.1 - _normalize_import_trace_options


### Test 2.2 - `_preflight_import_traces_request`

Qué prueba:
- caso feliz mínimo,
- fatal por `selected_fields` inválido,
- fatal por dtype `categorical` en `TraceSchema`,
- y fatal por `source_timezone` inválida.

In [8]:
# Caso feliz
issues = []
_preflight_import_traces_request(
    issues,
    df=RAW_POINTS,
    schema=BASE_IMPORT_SCHEMA,
    field_correspondence=RAW_TO_CANONICAL,
    options_eff=ImportTraceOptions(),
)
assert codes(issues) == []

# selected_fields inválido -> fatal
try:
    _preflight_import_traces_request(
        [],
        df=RAW_POINTS,
        schema=BASE_IMPORT_SCHEMA,
        field_correspondence=RAW_TO_CANONICAL,
        options_eff=ImportTraceOptions(selected_fields="location_category"),
    )
    raise AssertionError("Se esperaba fatal por selected_fields inválido")
except PylondrinaImportError:
    pass

# categorical no permitido -> fatal
schema_bad_categorical = make_trace_schema(
    [
        make_trace_field("point_id", "string", required=True),
        make_trace_field("user_id", "string", required=True),
        make_trace_field("time_utc", "datetime", required=True),
        make_trace_field("latitude", "float", required=True),
        make_trace_field("longitude", "float", required=True),
        make_trace_field("poi_type", "categorical", required=False),
    ],
    required=["point_id", "user_id", "time_utc", "latitude", "longitude"],
)
try:
    _preflight_import_traces_request(
        [],
        df=RAW_POINTS,
        schema=schema_bad_categorical,
        field_correspondence=RAW_TO_CANONICAL,
        options_eff=ImportTraceOptions(),
    )
    raise AssertionError("Se esperaba fatal por categorical en TraceSchema")
except SchemaError:
    pass

# source_timezone inválida -> fatal
try:
    _preflight_import_traces_request(
        [],
        df=RAW_POINTS,
        schema=BASE_IMPORT_SCHEMA,
        field_correspondence=RAW_TO_CANONICAL,
        options_eff=ImportTraceOptions(source_timezone="Not/A_Real_Timezone"),
    )
    raise AssertionError("Se esperaba fatal por source_timezone inválida")
except PylondrinaImportError:
    pass

show_ok("Test 2.2 - _preflight_import_traces_request")

OK - Test 2.2 - _preflight_import_traces_request


### Test 2.3 - `_resolve_trace_import_columns`

Qué prueba:
- renombrado efectivo con `field_correspondence`,
- preservación de extras cuando corresponde,
- descarte de extras cuando `keep_extra_fields=False`,
- y warning por `selected_fields` inexistentes.

In [9]:
# Caso 1: keep_extra_fields=True -> se preservan extras alcanzables
issues = []
work, applied, n_fields_mapped = _resolve_trace_import_columns(
    issues,
    RAW_POINTS,
    schema=BASE_IMPORT_SCHEMA,
    field_correspondence=RAW_TO_CANONICAL,
    options_eff=ImportTraceOptions(keep_extra_fields=True),
)
assert work.columns.tolist() == ["user_id", "time_utc", "latitude", "longitude", "location_category", "noise"]
assert applied == RAW_TO_CANONICAL
assert n_fields_mapped == len(RAW_TO_CANONICAL)
assert codes(issues) == []

# Caso 2: keep_extra_fields=False + selected_fields con un nombre inexistente
issues = []
work, applied, n_fields_mapped = _resolve_trace_import_columns(
    issues,
    RAW_POINTS,
    schema=BASE_IMPORT_SCHEMA,
    field_correspondence=RAW_TO_CANONICAL,
    options_eff=ImportTraceOptions(
        keep_extra_fields=False,
        selected_fields=["location_category", "missing_field"],
    ),
)
assert work.columns.tolist() == ["user_id", "time_utc", "latitude", "longitude", "location_category"]
assert applied == RAW_TO_CANONICAL
assert n_fields_mapped == len(RAW_TO_CANONICAL)
assert_has_code(issues, "IMP.OPTIONS.SELECTED_FIELDS_UNKNOWN")
assert_has_code(issues, "IMP.OPTIONS.EXTRA_FIELDS_DROPPED")

show_ok("Test 2.3 - _resolve_trace_import_columns")

OK - Test 2.3 - _resolve_trace_import_columns


### Test 2.4 - `_materialize_trace_core`

Qué prueba:
- generación automática de `point_id`,
- inserción en primera columna,
- y fatal cuando no se puede alcanzar el mínimo obligatorio de entrada.

In [10]:
# Genera point_id
df_no_point_id = pd.DataFrame(
    {
        "user_id": ["u1", "u2"],
        "time_utc": ["2026-01-01 08:00:00", "2026-01-01 09:30:00"],
        "latitude": [-33.45, -33.46],
        "longitude": [-70.66, -70.67],
    }
)

issues = []
work, point_id_generated = _materialize_trace_core(issues, df_no_point_id, strict=False)
assert point_id_generated is True
assert work.columns[0] == "point_id"
assert work["point_id"].tolist() == ["p0", "p1"]
assert_has_code(issues, "IMP.CORE.POINT_ID_GENERATED")

# Fatal si falta parte del mínimo no derivable
df_missing_core = pd.DataFrame(
    {
        "user_id": ["u1"],
        "time_utc": ["2026-01-01 08:00:00"],
        "latitude": [-33.45],
        # falta longitude
    }
)

try:
    _materialize_trace_core([], df_missing_core, strict=False)
    raise AssertionError("Se esperaba fatal por mínimo no alcanzable")
except PylondrinaImportError:
    pass

show_ok("Test 2.4 - _materialize_trace_core")

OK - Test 2.4 - _materialize_trace_core


### Test 2.5 - `_normalize_trace_time_utc`

Qué prueba:
- normalización a UTC usando `options.source_timezone`,
- warning por timezone no resuelta,
- y fatal por valores no parseables.

In [11]:
schema_no_tz = make_trace_schema(BASE_IMPORT_FIELDS, required=["point_id", "user_id", "time_utc", "latitude", "longitude"], timezone=None)

# Caso 1: source_timezone explícita
df_time = pd.DataFrame(
    {
        "point_id": ["p0", "p1"],
        "user_id": ["u1", "u2"],
        "time_utc": ["2026-01-01 08:00:00", "2026-01-01 09:30:00"],
        "latitude": [-33.45, -33.46],
        "longitude": [-70.66, -70.67],
    }
)

issues = []
norm, descriptor = _normalize_trace_time_utc(
    issues,
    df_time,
    schema=schema_no_tz,
    options_eff=ImportTraceOptions(source_timezone="-03:00"),
)
assert codes(issues) == []
assert descriptor["timezone_resolution"] == "options.source_timezone"
assert descriptor["normalized_to_utc"] is True
assert norm["time_utc"].iloc[0] == pd.Timestamp("2026-01-01 11:00:00")
assert norm["time_utc"].iloc[1] == pd.Timestamp("2026-01-01 12:30:00")

# Caso 2: parseable pero sin timezone resoluble -> warning
issues = []
norm, descriptor = _normalize_trace_time_utc(
    issues,
    df_time,
    schema=schema_no_tz,
    options_eff=ImportTraceOptions(source_timezone=None),
)
assert_has_code(issues, "IMP.TIME.TIMEZONE_UNRESOLVED")
assert descriptor["timezone_resolution"] == "unresolved"
assert descriptor["normalized_to_utc"] is False

# Caso 3: valores no parseables -> fatal
df_bad_time = df_time.copy()
df_bad_time.loc[1, "time_utc"] = "not-a-datetime"

try:
    _normalize_trace_time_utc(
        [],
        df_bad_time,
        schema=schema_no_tz,
        options_eff=ImportTraceOptions(source_timezone="UTC"),
    )
    raise AssertionError("Se esperaba fatal por time_utc no parseable")
except PylondrinaImportError:
    pass

show_ok("Test 2.5 - _normalize_trace_time_utc")

OK - Test 2.5 - _normalize_trace_time_utc


### Test 2.6 - `_finalize_import_traces_result`

Qué prueba:
- construcción final de `TraceDataset`,
- metadata mínima,
- evento `import_traces`,
- `ImportReport.summary`,
- `field_correspondence` aplicada,
- y JSON-safety del resultado.

In [12]:
df_final = pd.DataFrame(
    {
        "point_id": ["p0", "p1"],
        "user_id": ["u1", "u2"],
        "time_utc": pd.to_datetime(["2026-01-01 11:00:00", "2026-01-01 12:30:00"]),
        "latitude": [-33.45, -33.46],
        "longitude": [-70.66, -70.67],
        "location_category": ["home", "work"],
    }
)

issues = [
    Issue(level="info", code="IMP.CORE.POINT_ID_GENERATED", message="point_id generado"),
]

options_eff, parameters_effective = _normalize_import_trace_options(
    ImportTraceOptions(source_timezone="-03:00")
)

dataset, report = _finalize_import_traces_result(
    issues,
    df_final,
    schema=BASE_IMPORT_SCHEMA,
    source_name="raw_checkins",
    options_eff=options_eff,
    parameters_effective=parameters_effective,
    field_map_applied=RAW_TO_CANONICAL,
    n_fields_mapped=len(RAW_TO_CANONICAL),
    point_id_generated=True,
    temporal_descriptor={
        "time_field": "time_utc",
        "timezone_resolution": "options.source_timezone",
        "source_timezone_used": "-03:00",
        "schema_timezone": None,
        "normalized_to_utc": True,
    },
    provenance={"notebook": "helper_level_tests"},
    rows_in=2,
)

assert isinstance(dataset, TraceDataset)
assert isinstance(report, ImportReport)

assert dataset.metadata["is_validated"] is False
assert dataset.metadata["schema_version"] == BASE_IMPORT_SCHEMA.version
assert dataset.metadata["point_id_generated"] is True
assert dataset.metadata["source"] == {"name": "raw_checkins"}
assert dataset.metadata["field_correspondence_applied"]["user_id"] == "uid"
assert len(dataset.metadata["events"]) == 1
assert dataset.metadata["events"][0]["op"] == "import_traces"

assert report.summary == {
    "rows_in": 2,
    "rows_out": 2,
    "n_fields_mapped": len(RAW_TO_CANONICAL),
    "point_id_generated": True,
}
assert report.field_correspondence["user_id"] == "uid"
assert report.value_correspondence == {}
assert report.schema_version == BASE_IMPORT_SCHEMA.version

assert dataset.provenance == {"notebook": "helper_level_tests"}
assert_json_safe(dataset.metadata, "import_trace_metadata")
assert_json_safe(report.summary, "import_trace_report_summary")
assert_json_safe(dataset.metadata["events"][0], "import_trace_event")

show_ok("Test 2.6 - _finalize_import_traces_result")

OK - Test 2.6 - _finalize_import_traces_result


## Bloque 3. Tests de helpers principales de `validate_traces`

Qué se prueba en este bloque:
- normalización de options,
- preflight fatal y warnings de precheck,
- resolución de targets,
- required + types,
- constraints,
- monotonicidad temporal por usuario,
- y finalización del reporte/evento.

**Nota:** si el import de `pylondrina.validation_traces` falla al comienzo del notebook por el catálogo actual, corrígelo primero y luego ejecuta este bloque.

### Fixtures reutilizables mínimas para OP-15

In [13]:
BASE_VALIDATE_FIELDS = [
    make_trace_field("point_id", "string", required=True, constraints={"unique": True}),
    make_trace_field("user_id", "string", required=True),
    make_trace_field("time_utc", "datetime", required=True),
    make_trace_field("latitude", "float", required=True, constraints={"range": {"min": -90, "max": 90}}),
    make_trace_field("longitude", "float", required=True, constraints={"range": {"min": -180, "max": 180}}),
    make_trace_field("location_category", "string", required=False, constraints={"length": {"max": 20}}),
    make_trace_field("accuracy", "float", required=False, constraints={"nullable": False, "range": {"min": 0, "max": 100}}),
]

BASE_VALIDATE_SCHEMA = make_trace_schema(
    BASE_VALIDATE_FIELDS,
    required=["point_id", "user_id", "time_utc", "latitude", "longitude"],
)

VALID_TRACE_DF = pd.DataFrame(
    {
        "point_id": ["p0", "p1"],
        "user_id": ["u1", "u2"],
        "time_utc": ["2026-01-01T08:00:00", "2026-01-01T09:30:00"],
        "latitude": [-33.45, -33.46],
        "longitude": [-70.66, -70.67],
        "location_category": ["home", "work"],
        "accuracy": [10.0, 20.0],
    }
)

VALID_TRACES = make_trace_dataset(
    VALID_TRACE_DF,
    BASE_VALIDATE_SCHEMA,
    metadata={"dataset_id": "traces_001", "events": [], "is_validated": False},
)

show_ok("Fixtures de OP-15 listas")

OK - Fixtures de OP-15 listas


### Test 3.1 - `_normalize_trace_validation_options`

Qué prueba:
- defaults cuando `options=None`,
- preservación de valores explícitos,
- y forma efectiva lista para el pipeline.

In [14]:
opts = _normalize_trace_validation_options(None)
assert isinstance(opts, TraceValidationOptions)
assert opts.strict is False
assert opts.sample_rows_per_issue == 5
assert opts.validate_required_fields is True
assert opts.validate_types_and_formats is True
assert opts.validate_constraints is True
assert opts.validate_monotonic_time_per_user is True

opts = _normalize_trace_validation_options(
    TraceValidationOptions(
        strict=True,
        sample_rows_per_issue=3,
        validate_required_fields=True,
        validate_types_and_formats=False,
        validate_constraints=False,
        validate_monotonic_time_per_user=True,
    )
)
assert opts.strict is True
assert opts.sample_rows_per_issue == 3
assert opts.validate_types_and_formats is False
assert opts.validate_constraints is False

show_ok("Test 3.1 - _normalize_trace_validation_options")

OK - Test 3.1 - _normalize_trace_validation_options


### Test 3.2 - `_preflight_validate_traces_request`

Qué prueba:
- caso feliz sin skips,
- warning + skip para constraint conocida pero mal formada,
- y fatal por dtype `categorical`.

In [15]:
# Caso feliz
issues = []
skipped = _preflight_validate_traces_request(
    issues,
    traces=VALID_TRACES,
    options_eff=TraceValidationOptions(),
)
assert skipped == {}
assert codes(issues) == []

# Constraint conocida pero mal formada -> warning + skip
schema_skip = make_trace_schema(
    BASE_VALIDATE_FIELDS + [
        make_trace_field("score", "float", required=False, constraints={"range": {"minimum": 0}})
    ],
    required=["point_id", "user_id", "time_utc", "latitude", "longitude"],
)
traces_skip = make_trace_dataset(
    VALID_TRACE_DF.assign(score=[-1.0, 2.0]),
    schema_skip,
    metadata={"events": [], "is_validated": False},
)

issues = []
skipped = _preflight_validate_traces_request(
    issues,
    traces=traces_skip,
    options_eff=TraceValidationOptions(),
)
assert "score" in skipped
assert "range" in skipped["score"]
assert_has_code(issues, "VAL.SCHEMA.CONSTRAINT_INVALID_FORMAT")

# categorical no permitido -> fatal
schema_bad = make_trace_schema(
    [
        make_trace_field("point_id", "string", required=True),
        make_trace_field("user_id", "string", required=True),
        make_trace_field("time_utc", "datetime", required=True),
        make_trace_field("latitude", "float", required=True),
        make_trace_field("longitude", "float", required=True),
        make_trace_field("poi_type", "categorical", required=False),
    ],
    required=["point_id", "user_id", "time_utc", "latitude", "longitude"],
)
traces_bad = make_trace_dataset(
    VALID_TRACE_DF[["point_id", "user_id", "time_utc", "latitude", "longitude"]].copy(),
    schema_bad,
    metadata={"events": [], "is_validated": False},
)

try:
    _preflight_validate_traces_request(
        [],
        traces=traces_bad,
        options_eff=TraceValidationOptions(),
    )
    raise AssertionError("Se esperaba fatal por categorical en TraceSchema")
except SchemaError:
    pass

show_ok("Test 3.2 - _preflight_validate_traces_request")

OK - Test 3.2 - _preflight_validate_traces_request


### Test 3.3 - `_resolve_trace_validation_targets`

Qué prueba:
- unión entre núcleo canónico y `schema.required`,
- `checked_fields` efectivos,
- `checks_executed`,
- y `effective_nullable`.

In [16]:
required_fields, checked_fields, checks_executed, effective_nullable = _resolve_trace_validation_targets(
    VALID_TRACES,
    options_eff=TraceValidationOptions(),
)

assert required_fields == ["point_id", "user_id", "time_utc", "latitude", "longitude"]
assert "accuracy" in checked_fields
assert "location_category" in checked_fields

assert checks_executed == {
    "required_fields": True,
    "types_and_formats": True,
    "constraints": True,
    "monotonic_time_per_user": True,
}

assert effective_nullable["point_id"] is False
assert effective_nullable["user_id"] is False
assert effective_nullable["time_utc"] is False
assert effective_nullable["latitude"] is False
assert effective_nullable["longitude"] is False
assert effective_nullable["accuracy"] is False
assert effective_nullable["location_category"] is True

show_ok("Test 3.3 - _resolve_trace_validation_targets")

OK - Test 3.3 - _resolve_trace_validation_targets


### Test 3.4 - `_check_trace_required_and_types`

Qué prueba:
- required faltante,
- nulos en required,
- valores no parseables por dtype,
- y no mutación del dataframe.

In [21]:
required_fields = ["point_id", "user_id", "time_utc", "latitude", "longitude"]
checked_fields = ["point_id", "user_id", "time_utc", "latitude", "longitude", "accuracy"]

# Caso 1: falta una columna requerida
issues = []
df_missing = VALID_TRACE_DF.drop(columns=["longitude"])
_check_trace_required_and_types(
    issues,
    df_missing,
    schema=BASE_VALIDATE_SCHEMA,
    required_fields=required_fields,
    checked_fields=checked_fields,
    options_eff=TraceValidationOptions(),
)
assert_has_code(issues, "VAL.REQUIRED.MISSING_COLUMN")

# Caso 2: nulo en required + tipo inválido
issues = []
df_bad = VALID_TRACE_DF.copy()
df_bad.loc[0, "user_id"] = None
df_bad.loc[1, "latitude"] = "not-a-float"
df_bad.loc[1, "time_utc"] = "not-a-datetime"
before = df_bad.copy(deep=True)

_check_trace_required_and_types(
    issues,
    df_bad,
    schema=BASE_VALIDATE_SCHEMA,
    required_fields=required_fields,
    checked_fields=checked_fields,
    options_eff=TraceValidationOptions(sample_rows_per_issue=2),
)

assert_has_code(issues, "VAL.REQUIRED.NULL_IN_REQUIRED")
assert_has_code(issues, "VAL.TYPES.UNPARSEABLE_VALUE")
pd.testing.assert_frame_equal(df_bad, before)

show_ok("Test 3.4 - _check_trace_required_and_types")

OK - Test 3.4 - _check_trace_required_and_types


C:\Users\sebam\AppData\Local\Temp\ipykernel_17448\3706275049.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'not-a-float' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_bad.loc[1, "latitude"] = "not-a-float"


### Test 3.5 - `_check_trace_constraints`

Qué prueba:
- nullable=false en campo no requerido,
- range,
- pattern,
- length,
- unique,
- todo con issues agregados por campo/regla.

In [22]:
schema_constraints = make_trace_schema(
    [
        make_trace_field("point_id", "string", required=True, constraints={"unique": True}),
        make_trace_field("user_id", "string", required=True),
        make_trace_field("time_utc", "datetime", required=True),
        make_trace_field("latitude", "float", required=True, constraints={"range": {"min": -90, "max": 90}}),
        make_trace_field("longitude", "float", required=True, constraints={"range": {"min": -180, "max": 180}}),
        make_trace_field("accuracy", "float", required=False, constraints={"nullable": False, "range": {"min": 0, "max": 50}}),
        make_trace_field("label", "string", required=False, constraints={"pattern": r"^[A-Z]{2}$", "length": {"min": 2, "max": 2}}),
        make_trace_field("sensor_id", "string", required=False, constraints={"unique": True}),
    ],
    required=["point_id", "user_id", "time_utc", "latitude", "longitude"],
)

df_constraints = pd.DataFrame(
    {
        "point_id": ["p0", "p1"],
        "user_id": ["u1", "u2"],
        "time_utc": ["2026-01-01T08:00:00", "2026-01-01T09:00:00"],
        "latitude": [-33.45, -33.46],
        "longitude": [-70.66, -70.67],
        "accuracy": [None, 80.0],       # nullable violation + range violation
        "label": ["A1", "TOO_LONG"],    # pattern + length
        "sensor_id": ["s1", "s1"],      # unique
    }
)

required_fields, _, _, effective_nullable = _resolve_trace_validation_targets(
    make_trace_dataset(df_constraints, schema_constraints, metadata={"events": []}),
    options_eff=TraceValidationOptions(),
)

issues = []
_check_trace_constraints(
    issues,
    df_constraints,
    schema=schema_constraints,
    required_fields=required_fields,
    effective_nullable=effective_nullable,
    skipped_constraints={},
    options_eff=TraceValidationOptions(sample_rows_per_issue=2),
)

assert codes(issues).count("VAL.CONSTRAINTS.VIOLATION") >= 4
fields_with_violations = {issue.field for issue in issues}
assert "accuracy" in fields_with_violations
assert "label" in fields_with_violations
assert "sensor_id" in fields_with_violations

show_ok("Test 3.5 - _check_trace_constraints")

OK - Test 3.5 - _check_trace_constraints


### Test 3.6 - `_check_trace_monotonic_time_per_user`

Qué prueba:
- no issue cuando la secuencia es no decreciente,
- y warning cuando el orden temporal observado retrocede dentro de un usuario.

In [23]:
# Caso ok: orden no decreciente, incluyendo empate
df_ok = pd.DataFrame(
    {
        "point_id": ["p0", "p1", "p2"],
        "user_id": ["u1", "u1", "u2"],
        "time_utc": ["2026-01-01T08:00:00", "2026-01-01T08:00:00", "2026-01-01T09:00:00"],
        "latitude": [-33.45, -33.45, -33.46],
        "longitude": [-70.66, -70.66, -70.67],
    }
)
issues = []
_check_trace_monotonic_time_per_user(
    issues,
    df_ok,
    options_eff=TraceValidationOptions(),
)
assert codes(issues) == []

# Caso con retroceso temporal
df_bad = pd.DataFrame(
    {
        "point_id": ["p0", "p1", "p2", "p3"],
        "user_id": ["u1", "u1", "u2", "u2"],
        "time_utc": ["2026-01-01T08:00:00", "2026-01-01T07:59:00", "2026-01-01T09:00:00", "2026-01-01T09:05:00"],
        "latitude": [-33.45, -33.45, -33.46, -33.46],
        "longitude": [-70.66, -70.66, -70.67, -70.67],
    }
)
issues = []
_check_trace_monotonic_time_per_user(
    issues,
    df_bad,
    options_eff=TraceValidationOptions(sample_rows_per_issue=2),
)
assert_has_code(issues, "VAL.TEMPORAL.NON_MONOTONIC_TIME")
assert issues[0].level == "warning"

show_ok("Test 3.6 - _check_trace_monotonic_time_per_user")

OK - Test 3.6 - _check_trace_monotonic_time_per_user


### Test 3.7 - `_finalize_trace_validation`

Qué prueba:
- construcción del `ConsistencyReport`,
- summary mínimo estable,
- actualización de `metadata["is_validated"]`,
- append del evento `validate_traces`,
- y consistencia entre `event["summary"]` y `report.summary`.

In [24]:
checks_executed = {
    "required_fields": True,
    "types_and_formats": True,
    "constraints": True,
    "monotonic_time_per_user": True,
}
checked_fields = ["point_id", "user_id", "time_utc", "latitude", "longitude", "accuracy"]

# Caso ok
traces_ok = make_trace_dataset(
    VALID_TRACE_DF,
    BASE_VALIDATE_SCHEMA,
    metadata={"dataset_id": "traces_001", "events": [], "is_validated": False},
)
issues_ok = [Issue(level="warning", code="VAL.TEST.WARNING", message="warning de prueba")]
report_ok = _finalize_trace_validation(
    issues_ok,
    traces_ok,
    options_eff=TraceValidationOptions(sample_rows_per_issue=3),
    checked_fields=checked_fields,
    checks_executed=checks_executed,
)

assert isinstance(report_ok, ConsistencyReport)
assert report_ok.summary["ok"] is True
assert traces_ok.metadata["is_validated"] is True
assert len(traces_ok.metadata["events"]) == 1
event_ok = traces_ok.metadata["events"][0]
assert event_ok["op"] == "validate_traces"
assert event_ok["summary"] == report_ok.summary
assert getattr(report_ok, "parameters", {}) == {}
assert_json_safe(event_ok, "validate_trace_event_ok")

# Caso con error
traces_err = make_trace_dataset(
    VALID_TRACE_DF,
    BASE_VALIDATE_SCHEMA,
    metadata={"dataset_id": "traces_002", "events": [], "is_validated": False},
)
issues_err = [Issue(level="error", code="VAL.REQUIRED.MISSING_COLUMN", message="error de prueba")]
report_err = _finalize_trace_validation(
    issues_err,
    traces_err,
    options_eff=TraceValidationOptions(sample_rows_per_issue=3),
    checked_fields=checked_fields,
    checks_executed=checks_executed,
)

assert report_err.summary["ok"] is False
assert traces_err.metadata["is_validated"] is False
assert len(traces_err.metadata["events"]) == 1
assert traces_err.metadata["events"][0]["summary"] == report_err.summary

show_ok("Test 3.7 - _finalize_trace_validation")

OK - Test 3.7 - _finalize_trace_validation
